# Predictive Anayltics: Support Vector Machines

In [1]:
import pandas as pd
import polars as pl
import numpy as np
import matplotlib as plt
import datetime
from sklearn.svm import SVC 
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

## Preparations

In [55]:
# "Settings" / Decisions for the training data

DATA_PATH_TRAIN = "../data/train_test_data/train.parquet"
DATA_PATH_VAL = "../data/train_test_data/val.parquet"
DATA_PATH_TEST = "../data/train_test_data/test.parquet"

MODEL_PATH = "../models"

# Target and feature selection
TARGET_COL = "trip_demand"
EXCLUDE_COLS = [
    TARGET_COL,
    "datetime_hour",      # real timestamp would lead to much leakage
    "_split_bucket",      # only used for splitting
    "month", # cyclic feature is used instead
    "weekday", # cyclic feature is used instead
    "hour", # cyclic feature is used instead
    # all other taxi data columns must be excluded to prevent leakage
    "trip_seconds_sum",
    "trip_seconds_mean",
    "trip_seconds_min",
    "trip_seconds_max",
    "trip_miles_sum",
    "trip_miles_mean",
    "trip_miles_min",
    "trip_miles_max",
    "fare_sum",
    "fare_mean",
    "fare_min",
    "fare_max",
    "tips_sum",
    "tips_mean",
    "tips_min",
    "tips_max",
    "tolls_sum",
    "tolls_mean",
    "tolls_min",
    "tolls_max",
    "extras_sum",
    "extras_mean",
    "extras_min",
    "extras_max",
    "trip_total_sum",
    "trip_total_mean",
    "trip_total_min",
    "trip_total_max",
    "most_common_payment_type",
    "trip_count",
    "date",
]

Load data and select features and target

In [21]:
# Load data
train = pl.scan_parquet(DATA_PATH_TRAIN)
val = pl.scan_parquet(DATA_PATH_VAL)
test = pl.scan_parquet(DATA_PATH_TEST)

In [25]:
train_df = train.collect()
val_df = val.collect()
test_df = test.collect()

#print(train_df["trip_start_timestamp"].min(), train_df["trip_start_timestamp"].max())
#print(val_df["trip_start_timestamp"].min(), val_df["trip_start_timestamp"].max())
#print(test_df["trip_start_timestamp"].min(), test_df["trip_start_timestamp"].max())

train_df = train_df.to_pandas()
val_df = val_df.to_pandas()
test_df = test_df.to_pandas()

In [26]:
train_df.head()
type(train_df)

pandas.core.frame.DataFrame

In [45]:
# prepare data
# calculate median to split in low/high demand
# when trip_count above 50 percent use "high", when below or equal to 50 percent low
train_median = train_df["trip_count"].median()
train_df["trip_demand"] = np.where(train_df["trip_count"] > train_median, "high", "low")

val_median = val_df["trip_count"].median()
val_df["trip_demand"] = np.where(val_df["trip_count"] > val_median, "high", "low")

test_median = test_df["trip_count"].median()
test_df["trip_demand"] = np.where(test_df["trip_count"] > test_median, "high", "low")

In [57]:
# Create X and y
feature_cols = [
    col for col in train_df.columns
    if col not in EXCLUDE_COLS
]

X_train = train_df[feature_cols]
y_train = train_df[TARGET_COL]

X_val = val_df[feature_cols]
y_val = val_df[TARGET_COL]

X_test = test_df[feature_cols]
y_test = test_df[TARGET_COL]


print("Features:", X_train.dtypes)
print("Target:", y_train.dtypes)

Features: month_sin         float64
month_cos         float64
weekday_sin       float64
weekday_cos       float64
hour_sin          float64
hour_cos          float64
tmpc              float64
relh              float64
sknt              float64
vsby              float64
p01m              float64
skyc1_BKN            int8
skyc1_CLR            int8
skyc1_FEW            int8
skyc1_OVC            int8
skyc1_SCT            int8
skyc1_VV             int8
is_holiday           int8
community_area      int64
food_drink        float64
landmark          float64
shop              float64
train_station     float64
dtype: object
Target: object


In [50]:
train_df

,datetime_hour,month,weekday,hour,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,...,extras_sum,extras_mean,extras_min,extras_max,trip_total_sum,trip_total_mean,trip_total_min,trip_total_max,most_common_payment_type,trip_demand
0,2024-01-11 07:00:00,1,4,7,0.000000,1.000000,0.433884,-0.900969,9.659258e-01,-2.588190e-01,...,0.0,0.0,0.0,0.0,87.64,43.82,32.50,55.14,Unknown,high
1,2024-01-11 02:00:00,1,4,2,0.000000,1.000000,0.433884,-0.900969,5.000000e-01,8.660254e-01,...,0.0,0.0,0.0,0.0,0.00,0.00,0.00,0.00,No trips,low
2,2024-01-08 11:00:00,1,1,11,0.000000,1.000000,0.000000,1.000000,2.588190e-01,-9.659258e-01,...,0.0,0.0,0.0,0.0,0.00,0.00,0.00,0.00,No trips,low
3,2024-02-20 06:00:00,2,2,6,0.500000,0.866025,0.781831,0.623490,1.000000e+00,6.123234e-17,...,0.0,0.0,0.0,0.0,0.00,0.00,0.00,0.00,No trips,low
4,2024-02-16 00:00:00,2,5,0,0.500000,0.866025,-0.433884,-0.900969,0.000000e+00,1.000000e+00,...,0.0,0.0,0.0,0.0,0.00,0.00,0.00,0.00,No trips,low
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1100997,2026-01-05 12:00:00,1,1,12,0.000000,1.000000,0.000000,1.000000,1.224647e-16,-1.000000e+00,...,0.0,0.0,0.0,0.0,29.00,29.00,29.00,29.00,Unknown,low
1100998,2026-01-09 09:00:00,1,5,9,0.000000,1.000000,-0.433884,-0.900969,7.071068e-01,-7.071068e-01,...,0.0,0.0,0.0,0.0,68.25,68.25,68.25,68.25,Cash,low
1100999,2025-12-27 02:00:00,12,6,2,-0.500000,0.866025,-0.974928,-0.222521,5.000000e-01,8.660254e-01,...,0.0,0.0,0.0,0.0,0.00,0.00,0.00,0.00,No trips,low
1101000,2026-01-09 12:00:00,1,5,12,0.000000,1.000000,-0.433884,-0.900969,1.224647e-16,-1.000000e+00,...,0.0,0.0,0.0,0.0,0.00,0.00,0.00,0.00,No trips,low


In [ ]:
# Train SVC 
model = SVC()
model.fit(X_train, y_train)

In [ ]:
# Make prediction 
y_pred = model.predict(X_test)

In [ ]:
y_pred

array([1, 0, 0, 1, 1, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1])

In [ ]:
# Evaluate Model
print(confusion_matrix(y_test,y_pred))

print(classification_report(y_test,y_pred))

[[ 8  0]
 [ 0 12]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         8
           1       1.00      1.00      1.00        12

    accuracy                           1.00        20
   macro avg       1.00      1.00      1.00        20
weighted avg       1.00      1.00      1.00        20



In [ ]:
# get support vectors
model.support_vectors_

array([[-1.53745194e-01,  1.29405829e+00],
       [ 7.18746403e-01, -8.09639306e-01],
       [-2.39138636e-03, -8.38817303e-01],
       [-2.80132784e+00, -1.12880063e+01],
       [-3.27599890e+00, -8.86878913e+00],
       [-4.16705662e+00, -1.11095444e+01]])